# Step 4 — Map & traffic validation (safe first run)


In [1]:
import pickle
import sys

import networkx as nx
import pandas as pd
from IPython.display import display

from routing.config import MAP_RADIUS_M, ORIGIN, DESTINATION, RESULTS_DIR
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph

print("Python:", sys.executable)
print("Configured map radius:", MAP_RADIUS_M, "m")

G = load_road_graph(radius_m=MAP_RADIUS_M)
assert isinstance(G, nx.MultiDiGraph) and G.is_directed()
print("OSM graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "directed edges")

Python: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Configured map radius: 2000 m
OSM graph: 2569 nodes, 5850 directed edges


## 1 — Load the matching traffic snapshot safely

In [2]:
snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"
snapshot = None
summary = {}
traffic_bundle = None

if snapshot_path.exists():
    with snapshot_path.open("rb") as file:
        traffic_bundle = pickle.load(file)

    if isinstance(traffic_bundle, dict):
        snapshot = traffic_bundle.get("snapshot")
        summary = traffic_bundle.get("summary", {}) or {}
    elif isinstance(traffic_bundle, nx.MultiDiGraph):
        snapshot = traffic_bundle
    else:
        raise TypeError(
            "Unsupported traffic_snapshot.pkl format: "
            f"{type(traffic_bundle).__name__}"
        )

    if not isinstance(snapshot, nx.MultiDiGraph):
        raise TypeError("Saved traffic snapshot is missing or is not a MultiDiGraph")

    same_nodes = set(G.nodes) == set(snapshot.nodes)
    same_edges = set(G.edges(keys=True)) == set(snapshot.edges(keys=True))
    if not (same_nodes and same_edges):
        raise ValueError(
            "Saved traffic snapshot does not match the configured OSM graph. "
            "Run Notebook 05 to collect a new snapshot for this graph."
        )

    print("Loaded saved snapshot:", snapshot_path)
    print("Snapshot timestamp:", traffic_bundle.get("saved_at_utc", "not recorded") if isinstance(traffic_bundle, dict) else "not recorded")
else:
    print("No saved snapshot found. A temporary offline snapshot will be created.")
    print("This offline snapshot is NOT saved and will NOT replace any real traffic data.")

if snapshot is None:
    snapshot, summary = collect_traffic_snapshot(
        G, client=None, max_requests=0, pause_s=0
    )
    print("Temporary offline snapshot ready; no TomTom request was made.")

assert set(G.edges(keys=True)) == set(snapshot.edges(keys=True))
print("Snapshot graph:", snapshot.number_of_nodes(), "nodes,", snapshot.number_of_edges(), "directed edges")

Loaded saved snapshot: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl
Snapshot timestamp: 2026-11-08T08:15:14.138093+00:00
Snapshot graph: 2569 nodes, 5850 directed edges


## 2 — Display the complete road network and traffic layers

In [3]:
traffic_widget, traffic_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
    max_snap_distance_m=350.0,
)

display(traffic_widget)
print("Traffic coverage:", traffic_state["traffic_coverage_percent"], "%")
print("Traffic counts:", traffic_state["traffic_counts"])
print("Double-click the map to move the origin, then run the routing cell again.")

Traffic coverage: 0.239 %
Traffic counts: {'red': 0, 'yellow': 5, 'clear': 9, 'missing': 5836, 'closed': 0}
Double-click the map to move the origin, then run the routing cell again.


## 3 — Inspect traffic coverage

In [4]:
status_counts = traffic_state["traffic_counts"]
total_edges = snapshot.number_of_edges()
observed_edges = sum(status_counts[name] for name in ("red", "yellow", "clear", "closed"))

coverage_df = pd.DataFrame([
    {"traffic_status": status, "directed_edges": count}
    for status, count in status_counts.items()
])
display(coverage_df)

print(f"Observed traffic: {observed_edges:,} / {total_edges:,} edges")
print(f"Observed coverage: {traffic_state['traffic_coverage_percent']:.3f}%")
print("Important: missing roads use fallback travel times; they are not real-time observations.")

,traffic_status,directed_edges
0,red,0
1,yellow,5
2,clear,9
3,missing,5836
4,closed,0


Observed traffic: 14 / 5,850 edges
Observed coverage: 0.239%
Important: missing roads use fallback travel times; they are not real-time observations.


## 4 — Prepare routing graphs (no algorithms are run here)

In [5]:
free_flow_time_graph = make_routing_graph(
    snapshot, objective="time", scenario="free_flow"
)
measured_time_graph = make_routing_graph(
    snapshot, objective="time", scenario="measured"
)
measured_distance_graph = make_routing_graph(
    snapshot, objective="distance", scenario="measured"
)

print("Free-flow time edges:", free_flow_time_graph.number_of_edges())
print("Measured-time edges:", measured_time_graph.number_of_edges())
print("Measured-distance edges:", measured_distance_graph.number_of_edges())
print("PASS — routing graphs prepared from the same saved snapshot")

Free-flow time edges: 5850
Measured-time edges: 5850
Measured-distance edges: 5850
PASS — routing graphs prepared from the same saved snapshot
